# Part 3: Intelligence Layers - RAG→CAG→CRAG

Progressive enhancement of retrieval quality:

1. **RAG**: Basic retrieve + generate (Claude)
2. **CAG**: Add semantic caching layer
3. **CRAG**: Add document grading + corrective retrieval

**Tech Stack:**
- Embeddings: OpenAI text-embedding-3-small
- LLM: Claude 3.5 Haiku
- Vector DB: Qdrant (hybrid search)

In [ ]:
import sys
from pathlib import Path
import json
import os
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent / "src"))

from prime_lands.config import load_config
from prime_lands.indexing.qdrant_indexer import QdrantIndexer
from prime_lands.services.rag_service import RAGService
from prime_lands.services.cag_service import CAGService
from prime_lands.services.crag_service import CRAGService
from prime_lands.chunking.base import Chunk
from prime_lands.logger import setup_logger

load_dotenv(Path.cwd().parent / ".env")
setup_logger(level="INFO")

# Verify API keys
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY in .env"
assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY in .env"

print("✓ Imports and API keys verified")

## Step 1: Load Chunks & Config

In [ ]:
cfg = load_config(Path.cwd().parent / "config.yaml")
data_dir = Path.cwd().parent / "data"

# Load chunks (using semantic from Part 2)
chunks_file = data_dir / "chunks" / "semantic_chunks.jsonl"
chunks = []

with open(chunks_file, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(Chunk(**json.loads(line)))

print(f"Loaded {len(chunks)} chunks")
print(f"Sample chunk: {chunks[0].text[:100]}...")

## Step 2: Create Qdrant Collection & Index

⚠️ **Make sure Qdrant is running**: `docker run -d --name qdrant -p 6333:6333 qdrant/qdrant`

In [ ]:
# Initialize indexer
indexer = QdrantIndexer(cfg)

# Create collection
collection_name = "primelands_semantic"
await indexer.create_collection(collection_name, force=True)

print(f"✓ Created collection: {collection_name}")

In [ ]:
# Index chunks (this will take a few minutes)
await indexer.index_chunks(chunks, collection_name, batch_size=32)

print(f"✓ Indexed {len(chunks)} chunks with hybrid vectors (dense + sparse)")

## Step 3: Test Basic RAG

In [ ]:
# Initialize RAG service
rag_service = RAGService(cfg, indexer)

# Test query
test_query = "What 3 bedroom properties are available in Colombo?"

result = await rag_service.query(test_query, collection_name=collection_name)

print(f"\n🔍 Query: {test_query}")
print(f"\n💬 Answer:\n{result.answer}")
print(f"\n📊 Stats:")
print(f"  - Latency: {result.latency_ms:.0f}ms")
print(f"  - Cost: ${result.cost:.4f}")
print(f"  - Retrieved chunks: {result.metadata['retrieved_chunks']}")
print(f"  - Tokens: {result.metadata['input_tokens']} in, {result.metadata['output_tokens']} out")

## Step 4: Test CAG (Cache-Augmented Generation)

In [ ]:
# Initialize CAG service
cag_service = CAGService(cfg, rag_service)

# Test repeated queries
queries = [
    "What 3 bedroom properties are available in Colombo?",
    "Show me 3BR houses in Colombo",  # Similar query
    "What properties have ocean views?",
    "What are some 3 bedroom options in Colombo?",  # Similar to first
]

for i, query in enumerate(queries):
    print(f"\n{'='*60}")
    print(f"Query {i+1}: {query}")
    
    result = await cag_service.query(query, collection_name=collection_name)
    
    print(f"\nCache hit: {result.metadata.get('cache_hit', False)}")
    print(f"Latency: {result.latency_ms:.0f}ms")
    print(f"Cost: ${result.cost:.4f}")
    
    if result.metadata.get('cache_hit'):
        print(f"Matched cached query: '{result.metadata['cached_query']}'")

# Show cache stats
print(f"\n📈 CAG Statistics:")
stats = cag_service.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

## Step 5: Test CRAG (Corrective RAG)

In [ ]:
# Initialize CRAG service
crag_service = CRAGService(cfg, indexer)

# Test with potentially ambiguous query
ambiguous_query = "What are the luxury amenities available?"

print(f"🔍 Query: {ambiguous_query}")
print("\n⏳ Running CRAG (may take longer due to document grading)...\n")

result = await crag_service.query(ambiguous_query, collection_name=collection_name)

print(f"\n💬 Answer:\n{result.answer}")
print(f"\n📊 CRAG Metadata:")
print(f"  - Corrections applied: {result.metadata['corrections_applied']}")
print(f"  - Avg document grade: {result.metadata['avg_document_grade']:.3f}")
print(f"  - Retrieved chunks: {result.metadata['retrieved_chunks']}")
print(f"  - Relevant chunks: {result.metadata['relevant_chunks']}")
print(f"  - Final query: {result.metadata['final_query']}")
print(f"  - Latency: {result.latency_ms:.0f}ms")
print(f"  - Cost: ${result.cost:.4f}")

## Step 6: Compare All Three Services

In [ ]:
import pandas as pd

# Test same query across all services
test_query = "What family-friendly properties are near schools?"

print(f"Testing query: '{test_query}'\n")

# RAG
rag_result = await rag_service.query(test_query, collection_name=collection_name)

# CAG (fresh cache)
cag_result = await cag_service.query(test_query, collection_name=collection_name)

# CRAG
crag_result = await crag_service.query(test_query, collection_name=collection_name)

# Comparison
comparison = pd.DataFrame([
    {
        "Service": "RAG",
        "Latency (ms)": rag_result.latency_ms,
        "Cost ($)": rag_result.cost,
        "Retrieved": rag_result.metadata['retrieved_chunks'],
        "Answer Length": len(rag_result.answer),
    },
    {
        "Service": "CAG",
        "Latency (ms)": cag_result.latency_ms,
        "Cost ($)": cag_result.cost,
        "Retrieved": len(cag_result.contexts),
        "Answer Length": len(cag_result.answer),
    },
    {
        "Service": "CRAG",
        "Latency (ms)": crag_result.latency_ms,
        "Cost ($)": crag_result.cost,
        "Retrieved": crag_result.metadata['relevant_chunks'],
        "Answer Length": len(crag_result.answer),
    },
])

print("\n📊 Service Comparison:")
comparison

---

## ✅ Part 3 Complete!

**Key Achievements:**
- ✓ Hybrid vector indexing (dense + sparse)
- ✓ Basic RAG with Claude generation
- ✓ CAG with semantic caching (cost savings on repeated queries)
- ✓ CRAG with document grading and corrective retrieval

**Next Steps:**
1. Proceed to `04_performance_arena.ipynb` for RAGAS evaluation
2. Document service trade-offs in engineering report